# Samuel Choi - Assignment 1 - Compsci 361
### Overview
For this assignment we are to implement a ML classifier based on decision trees to classify wine quality based on the given input features.

["fixed acidity";"volatile acidity";"citric acid";"residual sugar";"chlorides";"free sulfur dioxide";"total sulfur dioxide";"density";"pH";"sulphates";"alcohol"]

Our model will be trained on these features to create a decision tree

# Task 1 (Coding)
### Overview
We have to implement a basic decision tree as discussed in the lecture.

First we will implement a *train* procedure for the decision tree.

Then we have to implement tree depth control as a means of stopping the model from growing too complex. We will have to implement a *stopping_depth* parameter in the train procedure.
We will then have to discuss the output of the training data at stopping levels 2, 3, and 4 respectively.

Lastly we will have to implement a test procedure for the complete decision tree algorithm. This procedure will take new data and our trained decision tree and returns a prediction for each example. We will then have to evaluate our predictions using one of the classifer evaluation methods we discussed in the lectures.

### Implementation

In [ ]:
import pandas as pd
import numpy as np


class Node:

    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

# Set a random seed to make the experiment reproducible
np.random.seed(44)

We first import pandas library for csv reading and parsing, and numpy for math-based operations in python. IPython will be useful for saving code output into Markdown.

The *Node* class will help us later when building the decision tree.

As per the requirements, we then set a random seed to make the experiment reproducible by anyone.

In [8]:
data = pd.read_csv('winequality-red.csv', sep=';')

data['quality'] = (data['quality'] >= 6).astype(int)

# Split the data in 80/20
train = data.sample(frac=0.8, random_state=44)
test = data.drop(train.index)

# We seperate the features and the labels
x_train = train.drop("quality", axis=1)
y_train = train["quality"]

x_test = test.drop("quality", axis=1)
y_test = test["quality"]

### Example data

In [5]:
import pandas as pd

data = data = pd.read_csv('winequality-red.csv', sep=';')
data['quality'] = (data['quality'] >= 6).astype(int)
df = pd.DataFrame(data.head())
df

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,1
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,0


We then read the csv seperating each column via *;*. We then change the *quality* column to binary instead of the numeric 0-10 it is at now. I chose to split it at 6 because it intuitively splits the column into two seperate groups and it is the middle point of 0-10.
Quality >= 6 is 1 while quality < 6 is 0.

For a data-set size of 1600, I chose a split of 80%/20% with 80% of the data being used for training and 20% of the data being used for testing. I chose this split because it provides significant amount of data (1280) for the decision tree to learn from while reserving a decent amount (320) for an unbiased evaluation of the model.

Once the data is split into training data and testing data, we then seperate both *training* and *testing* data into features and labels. The features are the properties of the wine while the label is the quality of the wine.

In [9]:
def entropy(labels):
    values, counts = np.unique(labels, return_counts=True)
    prob = counts/counts.sum()

    return -np.sum(prob * np.log2(prob))


# Information gain measures how good a split is
def information_gain(parent, left, right):
    parent_entropy = entropy(parent)
    
    n = len(parent)
    n_left = len(left)
    n_right = len(right)

    # Calculate weighted entropy of both left and right child
    weighted_entropy = (
        (n_left / n) * entropy(left) +
        (n_right / n) * entropy(right)
    )

    return parent_entropy - weighted_entropy

I then move on to implementing the *entropy* and *information_gain* function. *Entropy* will tell us how mixed/uncertain the class labels are. *Information gain* measures how much a certain split reduces this uncertainty. 

These two functions are important because they help the decision tree select the most informative feature and threshold at each step to split on. By splitting at the highest information gain, we ensure that the data becomes less entropic as the tree grows, resulting in higher accuracy.

In [10]:
# Search for best feature and threshold to split on
def best_split(X, y):
    best_gain = 0
    best_feature = None
    best_threshold = None

    for feature in X.columns:
        thresholds = X[feature].unique()

        for threshold in thresholds:

            left_mask = X[feature] <= threshold
            right_mask = X[feature] > threshold

            if sum(left_mask) == 0 or sum(right_mask) == 0:
                continue

            gain = information_gain(
                y, y[left_mask], y[right_mask]
            )

            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold

The *best_split* function then, for each feature, goes through all the given values and searches for the value that gives the highest *information gain* by "splitting" the data-set into lower than or greater than the threshold then calculating the information gain. It then returns that best feature and best value to create a decision tree node later.

In [11]:
def train_tree(X, y, depth=0, stopping_depth=None):

    # Stopping condition
    if len(np.unique(y)) == 1:
        return Node(value=y.iloc[0])
    
    if stopping_depth is not None and depth >= stopping_depth:
        return Node(value=y.mode()[0])
    
    feature, threshold = best_split(X, y)

    if feature is None:
        return Node(value=y.mode()[0])

    left_mask = X[feature] <= threshold
    right_mask = X[feature] > threshold

    # A check to prevent infinite recursion
    if left_mask.sum() == 0 or right_mask.sum() == 0:
        return Node(value=y.mode()[0])

    left_child = train_tree(
        X[left_mask], y[left_mask], depth+1, stopping_depth
    )

    right_child = train_tree(
        X[right_mask], y[right_mask], depth+1, stopping_depth
    )

    return Node(feature, threshold, left_child, right_child)

The *train_tree* function builds the full decision tree. This function uses a greedy recursive algorithm to build the tree. It recursively splits via best feature and best threhold using highest *information_gain* in the *best_split* function

The decision tree is a series of if-else rules where each internal node is a condition based on features, and each leaf node represents the expected value. These if-else rules were determined based on the best threshold determined through *best_split* function.

In [ ]:
def print_tree(node, depth=0):

    indent = "  " * depth

    if node.value is not None:
        print(indent + f"Predict: {node.value}")
        return

    print(indent + f"If {node.feature} <= {node.threshold}:")

    print_tree(node.left, depth + 1)

    print(indent + f"Else ({node.feature} > {node.threshold}):")

    print_tree(node.right, depth + 1)

The *print_tree* function will help us visualise the decision tree to find out how it was made.

## Tree Depth Control

Control of the tree depth is important to control model complexity. A tree with no depth limit might become too large such that overfitting is unavoidable, while a tree with a low depth limit might be too small such that underfitting becomes an issue.

### Output of the *train_tree* function using *stopping_depth = None*

In [14]:
tree = train_tree(x_train, y_train)

print_tree(tree)

If alcohol <= 10.2:
  If total sulfur dioxide <= 98.0:
    If volatile acidity <= 0.65:
      If sulphates <= 0.57:
        If chlorides <= 0.094:
          If total sulfur dioxide <= 88.0:
            If alcohol <= 9.8:
              If total sulfur dioxide <= 39.0:
                If citric acid <= 0.18:
                  If residual sugar <= 1.6:
                    Predict: 0
                  Else (residual sugar > 1.6):
                    If density <= 0.997:
                      Predict: 1
                    Else (density > 0.997):
                      If fixed acidity <= 7.8:
                        Predict: 1
                      Else (fixed acidity > 7.8):
                        Predict: 0
                Else (citric acid > 0.18):
                  If free sulfur dioxide <= 9.0:
                    If residual sugar <= 1.5:
                      Predict: 1
                    Else (residual sugar > 1.5):
                      Predict: 0
                  Else (free sul

From the above output, the decision tree is big with lots of conditions. This runs the risk of overfitting where the tree memorises the training_data rather than learning general rules.

### Output of the *train_tree* function using *stopping_depth = 2*

In [15]:
tree2 = train_tree(x_train, y_train, stopping_depth=2)

print_tree(tree2)

If alcohol <= 10.2:
  If total sulfur dioxide <= 98.0:
    Predict: 0
  Else (total sulfur dioxide > 98.0):
    Predict: 0
Else (alcohol > 10.2):
  If sulphates <= 0.58:
    Predict: 1
  Else (sulphates > 0.58):
    Predict: 1


From the above output, the decision tree with *stopping_depth = 2* is small. This decision tree might be too small where it runs the risk of underfitting, being too simple to fully capture the rules in the data.

### Output of the *train_tree* function using *stopping_depth = 3*

In [16]:
tree3 = train_tree(x_train, y_train, stopping_depth=3)

print_tree(tree3)

If alcohol <= 10.2:
  If total sulfur dioxide <= 98.0:
    If volatile acidity <= 0.65:
      Predict: 0
    Else (volatile acidity > 0.65):
      Predict: 0
  Else (total sulfur dioxide > 98.0):
    If pH <= 2.93:
      Predict: 1
    Else (pH > 2.93):
      Predict: 0
Else (alcohol > 10.2):
  If sulphates <= 0.58:
    If alcohol <= 11.4:
      Predict: 0
    Else (alcohol > 11.4):
      Predict: 1
  Else (sulphates > 0.58):
    If total sulfur dioxide <= 89.0:
      Predict: 1
    Else (total sulfur dioxide > 89.0):
      Predict: 0


A tree with a *stopping_depth* of 3 is noticably more complex than the tree with *stopping_depth = 2*. It is also noticably less complex than the tree with no *stopping_depth*.

### Output of the *train_tree* function with *stopping_depth = 4*

In [17]:
tree4 = train_tree(x_train, y_train, stopping_depth=4)

print_tree(tree4)

If alcohol <= 10.2:
  If total sulfur dioxide <= 98.0:
    If volatile acidity <= 0.65:
      If sulphates <= 0.57:
        Predict: 0
      Else (sulphates > 0.57):
        Predict: 1
    Else (volatile acidity > 0.65):
      If density <= 0.99814:
        Predict: 0
      Else (density > 0.99814):
        Predict: 0
  Else (total sulfur dioxide > 98.0):
    If pH <= 2.93:
      Predict: 1
    Else (pH > 2.93):
      If chlorides <= 0.066:
        Predict: 0
      Else (chlorides > 0.066):
        Predict: 0
Else (alcohol > 10.2):
  If sulphates <= 0.58:
    If alcohol <= 11.4:
      If volatile acidity <= 0.34:
        Predict: 1
      Else (volatile acidity > 0.34):
        Predict: 0
    Else (alcohol > 11.4):
      If citric acid <= 0.4:
        Predict: 1
      Else (citric acid > 0.4):
        Predict: 1
  Else (sulphates > 0.58):
    If total sulfur dioxide <= 89.0:
      If alcohol <= 11.3:
        Predict: 1
      Else (alcohol > 11.3):
        Predict: 1
    Else (total sulf

The tree with a *stopping_depth* of 4 is more complex than the others bar the tree with no *stopping_depth*.

## Testing the tree

We will run the trained trees on the *test* data. Our tree will make predictions for each data point and we will then evaluate the predictions using Confusion Matrix, Accuracy, Precision, Recall, and the F-measure.

In [20]:
def predict_sample(node, sample):
    if node.value is not None:
        return node.value
    
    if sample[node.feature] <= node.threshold:
        return predict_sample(node.left, sample)
    else:
        return predict_sample(node.right, sample)
    

def predict(tree, X):
    predictions = []

    for _, row in X.iterrows():
        predictions.append(predict_sample(tree, row))

    return np.array(predictions)

The *predict_sample* is the main function that will compare the features in each data point to the thresholds in the tree to choose how to traverse the decision tree.

Predict will just loop through the *test* data to create a prediction for each data point.

In [19]:
def evaluate_model(predictions):
    # Confusion Matrix
    TP = np.sum((predictions == 1) & (y_test == 1))
    TN = np.sum((predictions == 0) & (y_test == 0))
    FP = np.sum((predictions == 1) & (y_test == 0))
    FN = np.sum((predictions == 0) & (y_test == 1))

    # Accuracy, Precision, and Recall values
    accuracy = (TP + TN) / (TP + TN + FP + FN)

    precision = TP / (TP + FP) if (TP + FP) != 0 else 0

    recall = TP / (TP + FN) if (TP + FN) != 0 else 0

    # F-measure
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) != 0 else 0

    # Print results

    print("Confusion Matrix:")
    print("TP:", TP, "FP:", FP)
    print("FN:", FN, "TN:", TN)

    print("\nMetrics:")
    print("Accuracy:", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1 Score:", round(f1, 4))

The *evaluation_model* function will then take the prediction and print them out in a nice format to evaluate. The *evaluate_model* includes a Confusion Matrix, Accuracy, Precision, Recall, and a F-Measure.

### Evaluation of *stopping_depth = None*

In [22]:
predictions = predict(tree, x_test)

evaluate_model(predictions)

Confusion Matrix:
TP: 132 FP: 31
FN: 37 TN: 120

Metrics:
Accuracy: 0.7875
Precision: 0.8098
Recall: 0.7811
F1 Score: 0.7952


With a decision tree with *stopping_depth=None*, we can see that the model has correctly identified 132 true positives TP and 120 true negatives TN. However the model misidentified 37 false negaives FN, and 31 false positives FP.

This model correctly predicts 0.7947 or 79.47% of quality 0 wines to be quality 0. This model also correctly predicts 0.7811 or 78.11% of quality 1 wines to be quality 1. Overall this model is very balanced at predicting both quality 0 and quality 1 wines.

Accuracy, the percentage of test tuples that were correctly identified, is 0.7875 or 78.75%.

Precision, a measure of the accuracy of positive predictions, is 0.8098 or 80.98%.

Recall, a measure of how many actual positives were found, is 0.7811 or 78.11%.

Meanwhile the F1 score, a measure of the model's overall accuracy, is 0.7952 or 79.52%.

With this *stopping_depth* we can see that the model is reasonably accurate for it's task correctly identifying ~80% of data points correctly or 4 in 5 data points.

### Evaluation of *stopping_depth = 2*

In [23]:
predictions = predict(tree2, x_test)

evaluate_model(predictions)

Confusion Matrix:
TP: 114 FP: 40
FN: 55 TN: 111

Metrics:
Accuracy: 0.7031
Precision: 0.7403
Recall: 0.6746
F1 Score: 0.7059


With a decision tree with *stopping_depth=2*, we can see that the model has correctly identified 114 true positives TP and 111 true negatives TN. However the model misidentified 55 false negaives FN, and 40 false positives FP.

This model correctly predicts 0.7350 or 73.50% of quality 0 wines to be quality 0. This model also correctly predicts 0.6746 or 67.46% of quality 1 wines to be quality 1. This model is then very balanced at predicting both quality 0 and quality 1 wines.

Accuracy is 0.7031 or 70.31%.

Precision is 0.7403 or 74.03%.

Recall is 0.6746 or 67.46%.

Meanwhile the F1 score is 0.7059 or 70.59%.

With this *stopping_depth*, the model gets less accurate than compared to the model with no *stopping_depth* sitting at an overall accuracy of ~70%.

### Evaluation of *stopping_depth = 3*

In [24]:
predictions = predict(tree3, x_test)

evaluate_model(predictions)

Confusion Matrix:
TP: 97 FP: 28
FN: 72 TN: 123

Metrics:
Accuracy: 0.6875
Precision: 0.776
Recall: 0.574
F1 Score: 0.6599


With a decision tree with *stopping_depth=3*, we can see that the model has correctly identified 97 true positives TP and 123 true negatives TN. However the model misidentified 72 false negaives FN, and 28 false positives FP.

This model correctly predicts 0.8146 or 81.46% of quality 0 wines to be quality 0. However this model only correctly predicts 0.4260 or 42.60% of quality 1 wines to be quality 1.
This means this model is around 2x better at predicting quality 0 wines than quality 1 wines.

This is shocking because both the *stopping_depth=None* model and the *stopping_depth=2* model have an even distribution of prediction rates between quality 0 and 1 wines.

Accuracy is 0.6875 or 68.75%.

Precision is 0.776 or 77.6%.

Recall is 0.574 or 57.4%.

Meanwhile the F1 score is 0.6599 or 65.99%.

With this *stopping_depth*, the model gets more inaccurate than *stopping_depth=2*, sitting at an overall accuracy of ~65%. However if we take this model to just predict quality 0 then the overall accuracy would lie around ~81%.

### Evaluation of *stopping_depth = 4*

In [25]:
predictions = predict(tree4, x_test)

evaluate_model(predictions)

Confusion Matrix:
TP: 136 FP: 57
FN: 33 TN: 94

Metrics:
Accuracy: 0.7188
Precision: 0.7047
Recall: 0.8047
F1 Score: 0.7514


With a decision tree with *stopping_depth=4*, we can see that the model has correctly identified 136 true positives TP and 94 true negatives TN. However the model misidentified 33 false negaives FN, and 57 false positives FP.

This model correctly predicts 0.6225 or 62.25% of quality 0 wines to be quality 0. This model also correctly predicts 0.7907 or 79.07% of quality 1 wines to be quality 1. This model is somewhat better at correctly predicting quality 1 wines than quality 0 wines.

Accuracy is 0.7188 or 71.88%.

Precision is 0.7047 or 70.47%.

Recall is 0.8047 or 80.47%.

Meanwhile the F1 score is 0.7514 or 75.14%.

With this *stopping_depth*, the model becomes more accurate than the previous *stopping_depth* of 3, sitting at an overall accuracy of ~75%.

### Overall Evaluation of the 4 models

After building and testing all 4 models with the same testing data, the conclusion has to be made that the model with no *stopping_depth* is the best sitting at an overall accuracy of ~80%. The final ranking of prediction accuracy ranked from highest to lowest overall accuracy is:

1. *stopping_depth=None*
2. *stopping_depth=4*
3. *stopping_depth=2*
4. *stopping_depth=3*

However it is worth noting that *stopping_depth=3* does exceptionally well at correctly predicting quality 0 wines, sitting at an overall accuracy of *~81%*, which would be no. 1, if we only count quality 0 prediction accuracy.

If time and cost is not an issue, then *stopping_depth=None* is the best model to use with the highest overall accuracy of *~80%*. However if time and cost is an issue, then *stopping_depth=4* with an overall accuracy of *~75%* is recommended. If however, only quality 0 wines need to be predicted, then *stopping_depth=3* is recommended, sitting at quality 0 prediction rate of *~81%*.

# Task 2 (Reflection)